# ADS v2: Motor MTA / cancellation / arrears / claims / document events

Builds 44 event families: `M1`, `M1.F1`, `M.V1`, `M.V2`, `M.V3`, `M.V4`, `M3`, `M4`, `M4a`, `M6`, `M3.B1.1`, `M3.B1.2`, `M3.B1.4`, `M3.B1.5`, `M3.B1.3`, `M3.B1.6`, `C1a`, `C1b`, `C1a.F1`, `C2`, `C3`, `C4`, `C5`, `C6`, `ARR.1`, `ARR.2`, `ARR.O1`, `ARR.3`, `ARR.O2`, `CL1a`, `CL1b`, `CL1c`, `CL3.F1`, `CL4`, `CL.O1`, `CL.O2`, `D1`, `D1.F1`, `D3.F1`, `D.V2`, `D.V3`, `D.V4`, `D.V5`, `D.V6`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("15_Motor_InLife", "started")


## Shared: motor_inlife_calls

Inbound voice calls on the in-life queues (vehicle change, claims, documents out, cancel policy) or wrap-up coded as a claim on any queue (QD_* out of scope), summarised per conversation with the caller's phone key, the queue groups it reached, waits / abandons per group and the wrap-up categories. The caller's open policies at the call date (D-E1.24: the SCV key on the caller's number at the call time, then that customer's open policies, from 10_Common) give the product: 04 split Motor from Van with a scratch phone list (stg.VM0_van_phone_numbers, R-1) and matched claims / document callers through stg.Motor_PhoneNumbers (not built in 04).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_call_session                                                           */
/*  Description : Inbound voice sessions on the in-life queues, or wrap-up coded as a claim on any non- */
/*              : QD queue, in the window.                                                              */
/*              :                                                                                       */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_call_session as
select  ConversationId,
        SessionIndex,
        QueueName,
        trim(WrapupCodeName)                                            as WrapupCode,
        case when nullif(trim(WrapupCodeName), '') is null
               or trim(WrapupCodeName) in ('ININ-WRAP-UP-TIMEOUT', 'ININ-WRAP-UP-DELETED', 'Default Wrap-Up Code', '..')
             then 0 else 1 end                                          as IsCoded,
        ConversationStartTime,
        AniPhoneKey,
        AcdWaitDurationMs,
        Abandoned
from    ${ads.ods}.CallSession
where   MediaType = 'voice'
and     OriginatingDirection = 'inbound'
and     coalesce(left(QueueName, 3), '') <> 'QD_'
and     (QueueName in ('INBOUND_VehicleChangeAddress', 'INBOUND_VehicleChangeOther', 'INBOUND_VehicleChange_Perm',
                       'INBOUND_VehicleChange_Temp', 'INBOUND_VehicleUpdateLicence', 'INBOUND_Vehicle_Add_Driver',
                       'INBOUND_Claims', 'INBOUND_Documents_Out', 'INBOUND_CancelPolicy')
         or trim(WrapupCodeName) in ('Claims', 'Breakdown', 'Windscreen', 'Claims / Windscreen / Breakdown'))
and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_call                                                                   */
/*  Description : One row per conversation: start, caller phone key, the queue groups reached, waits an */
/*              : d abandons per group, the claim wrap-up codes, and the Documents_Out request type (D- */
/*              : E1.6: from the wrap-up code; Home-named codes and non-document codes flagged).        */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_minl_call_session                                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_call as
select  ConversationId,
        min(ConversationStartTime)                                      as CallStart,
        max(AniPhoneKey)                                                as CallerPhoneKey,
        max(case when QueueName in ('INBOUND_VehicleChangeAddress', 'INBOUND_VehicleChangeOther', 'INBOUND_VehicleChange_Perm',
                                    'INBOUND_VehicleChange_Temp', 'INBOUND_VehicleUpdateLicence', 'INBOUND_Vehicle_Add_Driver')
                 then 1 else 0 end)                                     as OnMta,
        max(case when QueueName in ('INBOUND_VehicleChangeOther', 'INBOUND_VehicleChange_Perm', 'INBOUND_VehicleChange_Temp')
                 then 1 else 0 end)                                     as OnMtaVehicle,
        max(case when QueueName = 'INBOUND_Vehicle_Add_Driver' then 1 else 0 end)   as OnMtaDriver,
        max(case when QueueName = 'INBOUND_VehicleUpdateLicence' then 1 else 0 end) as OnMtaLicence,
        max(case when QueueName = 'INBOUND_VehicleChangeAddress' then 1 else 0 end) as OnMtaAddress,
        sum(case when QueueName in ('INBOUND_VehicleChangeAddress', 'INBOUND_VehicleChangeOther', 'INBOUND_VehicleChange_Perm',
                                    'INBOUND_VehicleChange_Temp', 'INBOUND_VehicleUpdateLicence', 'INBOUND_Vehicle_Add_Driver')
                 then AcdWaitDurationMs else 0 end)                     as MtaWaitMs,
        sum(case when QueueName in ('INBOUND_VehicleChangeAddress', 'INBOUND_VehicleChangeOther', 'INBOUND_VehicleChange_Perm',
                                    'INBOUND_VehicleChange_Temp', 'INBOUND_VehicleUpdateLicence', 'INBOUND_Vehicle_Add_Driver')
                 then Abandoned else 0 end)                             as MtaAbandoned,
        max(case when QueueName = 'INBOUND_CancelPolicy' then 1 else 0 end)         as OnCancel,
        sum(case when QueueName = 'INBOUND_CancelPolicy' then AcdWaitDurationMs else 0 end) as CancelWaitMs,
        sum(case when QueueName = 'INBOUND_CancelPolicy' then Abandoned else 0 end) as CancelAbandoned,
        max(case when QueueName = 'INBOUND_Claims' then 1 else 0 end)               as OnClaims,
        max(case when QueueName = 'INBOUND_Claims' and IsCoded = 1 then 1 else 0 end) as ClaimsQueueCoded,
        max(case when WrapupCode = 'Claims' then 1 else 0 end)                      as CodeClaims,
        max(case when WrapupCode = 'Breakdown' then 1 else 0 end)                   as CodeBreakdown,
        max(case when WrapupCode = 'Windscreen' then 1 else 0 end)                  as CodeWindscreen,
        max(case when WrapupCode = 'Claims / Windscreen / Breakdown' then 1 else 0 end) as CodeClaimsMerged,
        max(case when QueueName = 'INBOUND_Documents_Out' then 1 else 0 end)        as OnDocsOut,
        sum(case when QueueName = 'INBOUND_Documents_Out' then AcdWaitDurationMs else 0 end) as DocsWaitMs,
        sum(case when QueueName = 'INBOUND_Documents_Out' then Abandoned else 0 end) as DocsAbandoned,
        max(case when QueueName <> 'INBOUND_Documents_Out' then null
                 when WrapupCode in ('NCB Request', 'Docs Out - NCB Request') then 'D.V2'
                 when WrapupCode in ('LOI Request (Motor)', 'LOI Request', 'Docs Out - LOI Request (Motor)', 'Docs Out - LOI Request') then 'D.V3'
                 when WrapupCode = 'CPF (Active) Payment Date/Amount Query' then 'D.V4'
                 when WrapupCode in ('LONDE Request', 'Docs Out - LONDE Request') then 'D.V5'
                 when WrapupCode in ('CS Documents Out', 'Cover Confirmation', 'Docs Out - Duplicate Pack', 'Docs Out - MTA Docs (Motor)') then 'D.V6'
            end)                                                        as DocRequestType,
        max(case when QueueName = 'INBOUND_Documents_Out'
                  and WrapupCode in ('C&D Request', 'Docs Out - C&D Request', 'Docs Out - MTA Docs',
                                     'NCB Request', 'Docs Out - NCB Request', 'LOI Request (Motor)', 'LOI Request',
                                     'Docs Out - LOI Request (Motor)', 'Docs Out - LOI Request',
                                     'CPF (Active) Payment Date/Amount Query', 'LONDE Request', 'Docs Out - LONDE Request',
                                     'CS Documents Out', 'Cover Confirmation', 'Docs Out - Duplicate Pack',
                                     'Docs Out - MTA Docs (Motor)')
                 then 1 else 0 end)                                     as DocsMotorCode,
        max(case when QueueName = 'INBOUND_Documents_Out'
                  and WrapupCode in ('LOI Home Request', 'Docs Out - LOI Home Request', 'Docs Out - Home renewal docs',
                                     'Docs Out - MTA Docs (Home)')
                 then 1 else 0 end)                                     as DocsHomeCode,
        max(case when QueueName = 'INBOUND_Documents_Out' and IsCoded = 1 then 1 else 0 end) as DocsCoded
from    ${ads.stg}.adsv2_minl_call_session
group by ConversationId

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_call_policy                                                            */
/*  Description : Per conversation, the caller's open policies by product at the call date (D-E1.24, sh */
/*              : ared step adsv2_call_open_policy: the SCV key holding the caller's number at the call */
/*              :  time, then that customer's Relay policies open at either end of the call's month). M */
/*              : otorPolicyCode is set only when exactly one Motor policy is open (never guessed, R-6) */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_minl_call, stg.adsv2_call_open_policy                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_call_policy as
select  c.ConversationId,
        coalesce(max(case when o.PolicyTypeGroup = 'Motor' then o.Policies end), 0)  as MotorPolicies,
        max(case when o.PolicyTypeGroup = 'Motor' and o.Policies = 1 then o.PolicyCode end) as MotorPolicyCode,
        max(case when o.PolicyTypeGroup = 'Motor' then o.PolicyId end)                as MotorPolicyId,
        coalesce(max(case when o.PolicyTypeGroup = 'Van' then o.Policies end), 0)    as VanPolicies,
        coalesce(max(case when o.PolicyTypeGroup = 'Home' then o.Policies end), 0)   as HomePolicies
from    ${ads.stg}.adsv2_minl_call c
join    ${ads.stg}.adsv2_call_open_policy o on o.ConversationId = c.ConversationId
group by c.ConversationId

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_motor_call                                                             */
/*  Description : In-life calls attributed to Motor: MTA-queue calls unless the caller holds only open  */
/*              : Van policies (04 dropped every caller whose phone matched any Van customer); claims a */
/*              : nd documents-out calls only when the caller holds an open Motor policy (as 04's phone */
/*              :  match to Motor policies); with the Motor policy when exactly one.                    */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_minl_call, stg.adsv2_minl_call_policy                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_motor_call as
select  c.*,
        coalesce(p.MotorPolicies, 0)                                    as MotorPolicies,
        coalesce(p.VanPolicies, 0)                                      as VanPolicies,
        coalesce(p.HomePolicies, 0)                                     as HomePolicies,
        p.MotorPolicyCode,
        case when p.MotorPolicyCode is not null then p.MotorPolicyId end as MotorPolicyId,
        case when coalesce(p.VanPolicies, 0) > 0 and coalesce(p.MotorPolicies, 0) = 0 then 0 else 1 end as MtaIsMotor,
        case when coalesce(p.MotorPolicies, 0) > 0 then 1 else 0 end    as CallerHasMotor
from    ${ads.stg}.adsv2_minl_call c
left join ${ads.stg}.adsv2_minl_call_policy p on p.ConversationId = c.ConversationId

## Shared: motor_mta_postings

MTA postings on Motor policies (04 cell 34, stg.JJulyMTAs: the accounts archive mapped to PremiumTypeGroup 'MTA' with 04's ar_* filter, kept exactly), one row per policy and posting date (an MTA posts its endorsement and fee lines the same day). 04 joined every month-end snapshot (stg.JulyPolicyState, 139 snapshots) and its 'not cancelled or lapsed' test then passed for any policy ever live, so no status filter is applied; the product comes from Policy.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_mta_posting                                                            */
/*  Description : Motor MTA postings in the window, per policy and posting date, with premium, fees and */
/*              :  whether any line moves money.                                                        */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyTransaction, ods_v2.Policy                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_mta_posting as
select  t.PolicyCode,
        max(p.PolicyId)                                                 as PolicyId,
        t.PostingDate,
        sum(coalesce(t.InsurerInitialNetPremium, 0) + coalesce(t.InsurerInitialCommission, 0)) as GrossPremium,
        sum(case when t.ClientTransType = '8' and t.DType = 5 then coalesce(t.ClientAmount, 0)
                 else coalesce(t.ClientFees, 0) end)                    as Fees,
        max(case when coalesce(t.InsurerInitialNetPremium, 0) + coalesce(t.InsurerInitialCommission, 0) <> 0
                   or (case when t.ClientTransType = '8' and t.DType = 5 then coalesce(t.ClientAmount, 0)
                            else coalesce(t.ClientFees, 0) end) <> 0
                 then 1 else 0 end)                                     as MovesMoney,
        max(case when t.PremiumType = 'Returns' then 1 else 0 end)      as HasReturn
from    ${ads.ods}.PolicyTransaction t
join    ${ads.ods}.Policy p     on p.PolicyCode = t.PolicyCode
where   p.PolicyTypeGroup = 'Motor'
and     t.PremiumTypeGroup = 'MTA'
and     (   (t.InsurerTransType in ('X', 'Y', 'Z', '1', '3', '5', 'C', 'E') and t.TransSeq = 0)
         or (t.ClientTransType = '1' and t.NewFlag in (0, 4) and t.InsurerStatus = '' and t.TransSeq = 0 and t.InsCoCode = 'ZZZZ')
         or (t.ClientTransType = '5' and t.ReturnSubType = '5' and t.TransSeq = 0 and t.InsCoCode = 'ZZZZ')
         or (t.ClientTransType = '8' and t.DType in (5, 6) and t.TransSeq = 0)
         or (t.InsCoCode = 'KCI' and t.AddonRef <> 0 and t.ShowOnReport = 'Y'))
and     cast(t.PostingDate as timestamp) >= timestamp '${ads.start_time}' and cast(t.PostingDate as timestamp) < timestamp '${ads.end_time}'
group by t.PolicyCode, t.PostingDate

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_mta_policy                                                             */
/*  Description : Motor policies with a Permanent MTA Relay document in the window (04 cell 32, stg.Mot */
/*              : orMTAPolicies).                                                                       */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_relay_document_event                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_mta_policy as
select  distinct PolicyCode
from    ${ads.stg}.adsv2_relay_document_event
where   PolicyTypeGroup = 'Motor'
and     EventDescription like 'Permanent%'
and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'

## M1: Customer calls Chill with a change request


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m1  (M1)                                                                */
/*  Description : Mta / Spine. Notebook 04 cell 281. Inbound calls that reached any of the six MTA queu */
/*              : es. Change: a caller holding only open Van policies at the call date is Van (VM1); 04 */
/*              :  dropped every caller whose phone matched any Van customer, so a Motor customer who a */
/*              : lso has a Van was lost. The Motor policy is filled when the caller holds exactly one. */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m1 as
select  xxhash64('M1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M1'                                                as EventTypeId,
        'Customer calls Chill with a change request' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                case when CallerPhoneKey is not null then 'PhoneKey' end        as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   OnMta = 1
        and     MtaIsMotor = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M1.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m1_f1  (M1.F1)                                                          */
/*  Description : Mta / Friction. Notebook 04 cells 52, 282. M1 calls that waited more than 600 seconds */
/*              :  on the MTA queues or were abandoned there (D-E1.25). 04 returned no rows: `not in` a */
/*              :  phone list holding a null.                                                           */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m1_f1 as
select  xxhash64('M1.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M1.F1'                                             as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                case when CallerPhoneKey is not null then 'PhoneKey' end        as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   OnMta = 1
        and     MtaIsMotor = 1
        and     (MtaWaitMs > 600000 or MtaAbandoned > 0)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M.V1: Vehicle change


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m_v1  (M.V1)                                                            */
/*  Description : Mta / Spine. Notebook 04 cell 287. M1 calls on the vehicle change queues (other, perm */
/*              : anent, temporary). The flow sheet marks M.V1-4 no signal; kept as 04.                 */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m_v1 as
select  xxhash64('M.V1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M.V1'                                              as EventTypeId,
        'Vehicle change' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                case when CallerPhoneKey is not null then 'PhoneKey' end        as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   OnMtaVehicle = 1
        and     MtaIsMotor = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M.V2: Named driver add or remove


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m_v2  (M.V2)                                                            */
/*  Description : Mta / Spine. Notebook 04 cell 288. M1 calls on the add-driver queue.                  */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m_v2 as
select  xxhash64('M.V2', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M.V2'                                              as EventTypeId,
        'Named driver add or remove' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                case when CallerPhoneKey is not null then 'PhoneKey' end        as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   OnMtaDriver = 1
        and     MtaIsMotor = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M.V3: Licence update


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m_v3  (M.V3)                                                            */
/*  Description : Mta / Spine. Notebook 04 cell 289. M1 calls on the licence update queue (written twic */
/*              : e on 2026-09-24 in 04; once here).                                                    */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m_v3 as
select  xxhash64('M.V3', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M.V3'                                              as EventTypeId,
        'Licence update' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                case when CallerPhoneKey is not null then 'PhoneKey' end        as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   OnMtaLicence = 1
        and     MtaIsMotor = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M.V4: Address change


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m_v4  (M.V4)                                                            */
/*  Description : Mta / Spine. Notebook 04 cell 290. M1 calls on the address change queue.              */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m_v4 as
select  xxhash64('M.V4', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M.V4'                                              as EventTypeId,
        'Address change' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                case when CallerPhoneKey is not null then 'PhoneKey' end        as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   OnMtaAddress = 1
        and     MtaIsMotor = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M3: Customer accepts


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m3  (M3)                                                                */
/*  Description : Mta / Spine. Notebook 04 cells 34, 283. Each MTA posting, at its posting date (date o */
/*              : nly). Change: one event per MTA (04 grouped by policy only, so a policy's second and  */
/*              : later MTAs over five years were lost, P15).                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyTransaction, Policy                    */
/*              : Reads: stg.adsv2_minl_mta_posting                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m3 as
select  xxhash64('M3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M3'                                                as EventTypeId,
        'Customer accepts' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(PostingDate as timestamp)                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_minl_mta_posting
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M4: Money due on the change


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m4  (M4)                                                                */
/*  Description : Mta / Spine. Notebook 04 cell 284. MTAs with a premium or fee line that is not zero ( */
/*              : as 04: return premiums count too), one per MTA at its posting date.                   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyTransaction, Policy                    */
/*              : Reads: stg.adsv2_minl_mta_posting                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m4 as
select  xxhash64('M4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M4'                                                as EventTypeId,
        'Money due on the change' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(PostingDate as timestamp)                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_minl_mta_posting
        where   MovesMoney = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M4a: Paid now


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m4a  (M4a)                                                              */
/*  Description : Mta / Spine. Notebook 04 cell 285. The same rows as M4, as 04 (P2: 'paid now' has no  */
/*              : payment-route test; decision needed).                                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyTransaction, Policy                    */
/*              : Reads: stg.adsv2_minl_mta_posting                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m4a as
select  xxhash64('M4a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M4a'                                               as EventTypeId,
        'Paid now' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(PostingDate as timestamp)                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_minl_mta_posting
        where   MovesMoney = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M6: Cert and disc sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m6  (M6)                                                                */
/*  Description : Mta / Spine. Notebook 04 cell 286. Emailed or transmitted Permanent MTA documents (ce */
/*              : rtificate, C&D letter, schedule ...) of policies with an MTA posting in the window, o */
/*              : ne event per policy and day at the first document. Change: 04 took the last emailed d */
/*              : ocument of any type (renewal and NB packs included) once per policy for five years.   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyTransaction       */
/*              : Reads: stg.adsv2_relay_document_event, stg.adsv2_minl_mta_posting                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m6 as
select  xxhash64('M6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M6'                                                as EventTypeId,
        'Cert and disc sent' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                d.PolicyCode                                                    as SourcePolicyReference,
                max(d.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_relay_document_event d
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_minl_mta_posting) m on m.PolicyCode = d.PolicyCode
        where   d.PolicyTypeGroup = 'Motor'
        and     d.EventDescription like 'Permanent MTA%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by d.PolicyCode, cast(d.EventDateTime as date)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M3.B1.1: Document request issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m3_b1_1  (M3.B1.1)                                                      */
/*  Description : Mta / Branch. Notebook 04 cells 32, 291. Motor policies with a Permanent MTA document */
/*              :  that are in the DAY 1 chase, at their latest sale date.                              */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan, PolicyDocumentEvent   */
/*              : Reads: ods_v2.DocumentChaseMotorVan, stg.adsv2_minl_mta_policy                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m3_b1_1 as
select  xxhash64('M3.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M3.B1.1'                                           as EventTypeId,
        'Document request issued' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan c
        join    ${ads.stg}.adsv2_minl_mta_policy m on m.PolicyCode = c.PolicyCode
        where   cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     c.PolicyTypeGroup = 'Motor'
        and     c.Campaign = 'DAY 1'
        group by c.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M3.B1.2: Customer submits documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m3_b1_2  (M3.B1.2)                                                      */
/*  Description : Mta / Branch. Notebook 04 cell 292. M3.B1.1 policies with a MyChill upload in the win */
/*              : dow, at the latest chase sale date (as 04).                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan, DocumentUpload, Polic */
/*              : yDocumentEvent                                                                        */
/*              : Reads: ods_v2.DocumentChaseMotorVan, stg.adsv2_minl_mta_policy, ods_v2.DocumentUpload */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m3_b1_2 as
select  xxhash64('M3.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M3.B1.2'                                           as EventTypeId,
        'Customer submits documents' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan c
        join    ${ads.stg}.adsv2_minl_mta_policy m on m.PolicyCode = c.PolicyCode
        join    (
                select  distinct PolicyCode
                from    ${ads.ods}.DocumentUpload
                where   cast(EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(EventTimestamp as timestamp) < timestamp '${ads.end_time}'
                ) u     on u.PolicyCode = c.PolicyCode
        where   cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     c.PolicyTypeGroup = 'Motor'
        and     c.Campaign = 'DAY 1'
        group by c.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M3.B1.4: Chase-  first reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m3_b1_4  (M3.B1.4)                                                      */
/*  Description : Mta / Branch. Notebook 04 cell 293. The same rule as M3.B1.1 (DAY 1), as 04 (P2; DAY  */
/*              : 7 would be the first reminder, as Home D-E1.16).                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan, PolicyDocumentEvent   */
/*              : Reads: ods_v2.DocumentChaseMotorVan, stg.adsv2_minl_mta_policy                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m3_b1_4 as
select  xxhash64('M3.B1.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M3.B1.4'                                           as EventTypeId,
        'Chase-  first reminder' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan c
        join    ${ads.stg}.adsv2_minl_mta_policy m on m.PolicyCode = c.PolicyCode
        where   cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     c.PolicyTypeGroup = 'Motor'
        and     c.Campaign = 'DAY 1'
        group by c.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M3.B1.5: Chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m3_b1_5  (M3.B1.5)                                                      */
/*  Description : Mta / Branch. Notebook 04 cell 294. Motor policies with a Permanent MTA document that */
/*              :  are in the DAY 20 chase.                                                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan, PolicyDocumentEvent   */
/*              : Reads: ods_v2.DocumentChaseMotorVan, stg.adsv2_minl_mta_policy                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m3_b1_5 as
select  xxhash64('M3.B1.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M3.B1.5'                                           as EventTypeId,
        'Chase-  escalation' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan c
        join    ${ads.stg}.adsv2_minl_mta_policy m on m.PolicyCode = c.PolicyCode
        where   cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     c.PolicyTypeGroup = 'Motor'
        and     c.Campaign = 'DAY 20'
        group by c.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M3.B1.3: Documents validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m3_b1_3  (M3.B1.3)                                                      */
/*  Description : Mta / Branch. Notebook 04 cell 295. The same rule as M3.B1.2, as 04 (P2: 'validated'  */
/*              : is not tested; decision needed).                                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan, DocumentUpload, Polic */
/*              : yDocumentEvent                                                                        */
/*              : Reads: ods_v2.DocumentChaseMotorVan, stg.adsv2_minl_mta_policy, ods_v2.DocumentUpload */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m3_b1_3 as
select  xxhash64('M3.B1.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M3.B1.3'                                           as EventTypeId,
        'Documents validated' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan c
        join    ${ads.stg}.adsv2_minl_mta_policy m on m.PolicyCode = c.PolicyCode
        join    (
                select  distinct PolicyCode
                from    ${ads.ods}.DocumentUpload
                where   cast(EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(EventTimestamp as timestamp) < timestamp '${ads.end_time}'
                ) u     on u.PolicyCode = c.PolicyCode
        where   cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     c.PolicyTypeGroup = 'Motor'
        and     c.Campaign = 'DAY 1'
        group by c.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## M3.B1.6: Documents received and validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_m3_b1_6  (M3.B1.6)                                                      */
/*  Description : Mta / Branch. Notebook 04 cell 296. Accepted MyChill workflow records of policies wit */
/*              : h an MTA posting in the window, one event per record (as A6.B1.6). 04 kept the last a */
/*              : ccepted record per policy over five years.                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentWorkflowStatus, PolicyTransaction    */
/*              : Reads: ods_v2.DocumentWorkflowStatus, stg.adsv2_minl_mta_posting                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_m3_b1_6 as
select  xxhash64('M3.B1.6', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'M3.B1.6'                                           as EventTypeId,
        'Documents received and validated' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(w.EventTimestamp as timestamp)                             as EventDateTime,
                1                                                               as SourceSystemId,
                w.WorkflowGuid                                                  as EventSourceReference,
                w.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.DocumentWorkflowStatus w
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_minl_mta_posting) m on m.PolicyCode = w.PolicyCode
        where   cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        and     w.IsAccepted = true
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## C1a: Customer asks to cancel


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_c1a  (C1a)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 297, rebuilt per R-5: customer-requested cance */
/*              : llations (Customer Request / Client reasons) at their cancel date, one per cancelled  */
/*              : policy. 04 found 113 of 2,389 (joined on the new CXL code).                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyCancellation                           */
/*              : Reads: stg.adsv2_minl_cancellation                                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_c1a as
select  xxhash64('C1a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'C1a'                                               as EventTypeId,
        'Customer asks to cancel' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CancelDateTime                                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_minl_cancellation
        where   ReasonGroup = 'Customer'
        and     CancelDateTime >= timestamp '${ads.start_time}' and CancelDateTime < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## C1b: Trigger detected


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_c1b  (C1b)                                                              */
/*  Description : Cancellation / Spine. New (R-5, D-E1.4): enforced / insurer-initiated cancellations ( */
/*              : the trigger detected), at the cancel date.                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyCancellation                           */
/*              : Reads: stg.adsv2_minl_cancellation                                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_c1b as
select  xxhash64('C1b', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'C1b'                                               as EventTypeId,
        'Trigger detected' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CancelDateTime                                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_minl_cancellation
        where   ReasonGroup = 'Enforced'
        and     CancelDateTime >= timestamp '${ads.start_time}' and CancelDateTime < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## C1a.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_c1a_f1  (C1a.F1)                                                        */
/*  Description : Cancellation / Friction. Notebook 04 cell 298 rebuilt (D-E1.25): calls on the cancel- */
/*              : policy queue by a caller with an open Motor policy that waited more than 600 seconds  */
/*              : there or were abandoned. 04 read a reason like '%NCT%' at one month end (no rows).    */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_c1a_f1 as
select  xxhash64('C1a.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'C1a.F1'                                            as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                case when CallerPhoneKey is not null then 'PhoneKey' end        as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   OnCancel = 1
        and     CallerHasMotor = 1
        and     (CancelWaitMs > 600000 or CancelAbandoned > 0)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## C2: Document chase-  reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_c2  (C2)                                                                */
/*  Description : Cancellation / Spine. Notebook 04 cell 299. Chase snapshot rows with one of six docum */
/*              : ents outstanding for the cancelled policy itself, sale date within 120 days before th */
/*              : e cancel date, at the earliest sale date; one per cancellation. 04 matched any policy */
/*              :  of any client in the cancellation table, whatever the date.                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan, PolicyCancellation    */
/*              : Reads: stg.adsv2_minl_cancellation, ods_v2.DocumentChaseMotorVan                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_c2 as
select  xxhash64('C2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'C2'                                                as EventTypeId,
        'Document chase-  reminder' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                x.PolicyCode                                                    as SourcePolicyReference,
                max(x.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_minl_cancellation x
        join    ${ads.ods}.DocumentChaseMotorVan c on c.PolicyCode = x.PolicyCode
        where   x.ReasonGroup <> 'Excluded'
        and     c.PolicyTypeGroup = 'Motor'
        and     (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O')
        and     coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') between date_sub(cast(x.CancelDateTime as date), 120) and cast(x.CancelDateTime as date)
        and     cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        group by x.PolicyCode, x.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## C3: Document chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_c3  (C3)                                                                */
/*  Description : Cancellation / Spine. Notebook 04 cell 300. As C2, on the later chase campaigns (not  */
/*              : DAY 1).                                                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan, PolicyCancellation    */
/*              : Reads: stg.adsv2_minl_cancellation, ods_v2.DocumentChaseMotorVan                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_c3 as
select  xxhash64('C3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'C3'                                                as EventTypeId,
        'Document chase-  escalation' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                x.PolicyCode                                                    as SourcePolicyReference,
                max(x.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_minl_cancellation x
        join    ${ads.ods}.DocumentChaseMotorVan c on c.PolicyCode = x.PolicyCode
        where   x.ReasonGroup <> 'Excluded'
        and     c.PolicyTypeGroup = 'Motor'
        and     (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O')
        and     c.Campaign <> 'DAY 1'
        and     coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') between date_sub(cast(x.CancelDateTime as date), 120) and cast(x.CancelDateTime as date)
        and     cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        group by x.PolicyCode, x.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## C4: Final notice


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_c4  (C4)                                                                */
/*  Description : Cancellation / Spine. Notebook 04 cells 53, 301. A 'Chase ... Final' Relay document o */
/*              : n the cancelled policy within 120 days before its cancel date, for a policy in the ne */
/*              : w-business DAY 20 chase with any document outstanding (04's stg.MotorEscalated20days, */
/*              :  second-car certificate column fixed); the latest such document per cancellation. 04  */
/*              : took any final chase of a DAY 20 policy, cancelled or not.                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, DocumentChaseMotorVan,  */
/*              : PolicyCancellation                                                                    */
/*              : Reads: stg.adsv2_minl_cancellation, stg.adsv2_relay_document_event, ods_v2.DocumentCh */
/*              : aseMotorVan                                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_c4 as
select  xxhash64('C4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'C4'                                                as EventTypeId,
        'Final notice' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                x.PolicyCode                                                    as SourcePolicyReference,
                max(x.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_minl_cancellation x
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = x.PolicyCode
        join    (
                select  distinct PolicyCode
                from    ${ads.ods}.DocumentChaseMotorVan
                where   cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
                and     PolicyType = 'New Business'
                and     PolicyTypeGroup = 'Motor'
                and     (Sign_Prop_Status = 'O' or Sign_Stat_Of_Fact_Status = 'O' or Comp_Fin_Agree_Status = 'O' or Comp_DDM_Status = 'O' or Proof_Of_NCB_Status = 'O' or Trans_Proof_Of_NCB_Status = 'O' or Ltr_Of_Driv_Exp_Status = 'O' or Prop_Driv_Lic_Status = 'O' or Nam_Driv_Lic_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or SecondCar_Cert_Status = 'O' or Doc_Ltr_Status = 'O' or Engineers_Rpt_Status = 'O' or NCT_Status = 'O' or Veh_Lic_Cert_Status = 'O' or Irish_Reg_Status = 'O' or Main_Driver_Dec_Status = 'O' or Soc_Dom_Pleas_Dec_Status = 'O' or Comp_Car_Exper_Status = 'O' or DD_Conf_Ltr_Status = 'O' or Comp_Fin_Mand_Status = 'O' or Val_For_Spec_Item_Status = 'O' or Dri_Lic_Num_Status = 'O' or Comp_Lost_Cert_Dec_Status = 'O' or Orig_Cert_Status = 'O' or Cancel_Req_Status = 'O' or PPS_Num_Status = 'O' or Afford_State_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O')
                and     Campaign = 'DAY 20'
                ) e     on e.PolicyCode = x.PolicyCode
        where   x.ReasonGroup <> 'Excluded'
        and     d.PolicyTypeGroup = 'Motor'
        and     upper(d.EventDescription) like '%CHASE%'
        and     upper(d.EventDescription) like '%FINAL%'
        and     d.EventDateTime between cast(date_sub(cast(x.CancelDateTime as date), 120) as timestamp) and x.CancelDateTime
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by x.PolicyCode, x.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## C5: Cancellation processed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_c5  (C5)                                                                */
/*  Description : Cancellation / Spine. Notebook 04 cell 302. Emailed or transmitted 'CXL' Relay docume */
/*              : nts on the cancelled policy within 30 days either side of its cancel date, the latest */
/*              :  per cancellation. 04 took the last CXL document of any policy of a client in the can */
/*              : cellation table.                                                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent, PolicyCancellation      */
/*              : Reads: stg.adsv2_minl_cancellation, stg.adsv2_relay_document_event                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_c5 as
select  xxhash64('C5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'C5'                                                as EventTypeId,
        'Cancellation processed' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                x.PolicyCode                                                    as SourcePolicyReference,
                max(x.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_minl_cancellation x
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = x.PolicyCode
        where   x.ReasonGroup <> 'Excluded'
        and     d.PolicyTypeGroup = 'Motor'
        and     d.EventDescription like '%CXL%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime between cast(date_sub(cast(x.CancelDateTime as date), 30) as timestamp)
                                    and cast(date_add(cast(x.CancelDateTime as date), 31) as timestamp)
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by x.PolicyCode, x.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## C6: Cancellation confirmation sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_c6  (C6)                                                                */
/*  Description : Cancellation / Spine. Notebook 04 cell 303. The insurer-led cancellation email (Chill */
/*              : 's spelling 'Cancelation' kept), one event per document. As 04, not tied to a cancell */
/*              : ation record (customer cancellations get no such email).                              */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyDocumentEvent                          */
/*              : Reads: stg.adsv2_relay_document_event                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_c6 as
select  xxhash64('C6', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'C6'                                                as EventTypeId,
        'Cancellation confirmation sent' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(EventDateTime)                                              as EventDateTime,
                1                                                               as SourceSystemId,
                cast(EventId as string)                                         as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                max(PolicyId)                                                   as PolicyId
        from    ${ads.stg}.adsv2_relay_document_event
        where   PolicyTypeGroup = 'Motor'
        and     EventDescription = 'Insurer Led Cancelation - Emailed Document - Reg canx email template'
        and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, EventId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## Shared: minl_arrears_episode

Motor arrears episodes (D-E1.9): each loan status run 5 (arrears) entered from 6 (active), with the next two runs of the same loan. Loan status meanings are read from the transitions (no code table; for client validation).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_arrears_episode                                                        */
/*  Description : One row per arrears run entered from active, Motor policies: start, the next status a */
/*              : nd its first day, and the status after that. EpisodeChaseEnd = the next run's first d */
/*              : ay, or the cancellation day when the loan defaults and is then cancelled (still open: */
/*              :  the last day seen + 1).                                                              */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyLoanStatus, ods_v2.Policy                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_arrears_episode as
select  e.LoanReference,
        e.PolicyId,
        e.PolicyCode,
        e.ClientCode,
        e.StatusFromDate                                                as ArrearsStart,
        e.NextLoanStatusId,
        e.NextFromDate,
        e.Next2LoanStatusId,
        e.Next2FromDate,
        case when e.NextLoanStatusId = 4 and e.Next2LoanStatusId = 3 then e.Next2FromDate
             else coalesce(e.NextFromDate, date_add(e.LastSeenDate, 1)) end as EpisodeChaseEnd
from    (
        select  l.*,
                lead(l.LoanStatusId, 1) over (partition by l.LoanReference order by l.StatusFromDate) as NextLoanStatusId,
                lead(l.StatusFromDate, 1) over (partition by l.LoanReference order by l.StatusFromDate) as NextFromDate,
                lead(l.LoanStatusId, 2) over (partition by l.LoanReference order by l.StatusFromDate) as Next2LoanStatusId,
                lead(l.StatusFromDate, 2) over (partition by l.LoanReference order by l.StatusFromDate) as Next2FromDate
        from    ${ads.ods}.PolicyLoanStatus l
        join    ${ads.ods}.Policy p     on p.PolicyCode = l.PolicyCode
        where   p.PolicyTypeGroup = 'Motor'
        ) e
where   e.LoanStatusId = 5
and     e.PreviousLoanStatusId = 6

## ARR.1: Missed payment detected


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_arr_1  (ARR.1)                                                          */
/*  Description : Arrears / Spine. D-E1.9: missed payment detected = the loan moves from active (6) to  */
/*              : arrears (5) in the Close Brothers file, at the first load day in arrears. 04 used the */
/*              :  Chase 1 SMS (the same rows as ARR.2).                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyLoanStatus                             */
/*              : Reads: stg.adsv2_minl_arrears_episode                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_arr_1 as
select  xxhash64('ARR.1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'ARR.1'                                             as EventTypeId,
        'Missed payment detected' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(ArrearsStart as timestamp)                                 as EventDateTime,
                24                                                              as SourceSystemId,
                LoanReference                                                   as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType,
                PolicyId
        from    ${ads.stg}.adsv2_minl_arrears_episode
        where   cast(ArrearsStart as timestamp) >= timestamp '${ads.start_time}' and cast(ArrearsStart as timestamp) < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## ARR.2: Arrears chase-  automated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_arr_2  (ARR.2)                                                          */
/*  Description : Arrears / Spine. D-E1.9: each 'Arrears - Chase 1..5' SMS sent (not Mop Up / TEST / Ch */
/*              : ase 6), one event per send, to the client of a Motor arrears episode from its start t */
/*              : o its end (cleared, defaulted, or escalated to cancellation). Most sends carry the cl */
/*              : ient code only; the episode gives the policy. 04 counted the Chase 1 send only, once  */
/*              : per policy of the client's current policies.                                          */
/*              :                                                                                       */
/*              : Grain: Customer   Product: Motor   Source: MessageEvent, PolicyLoanStatus             */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_minl_arrears_episode                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_arr_2 as
select  xxhash64('ARR.2', 'Customer',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'ARR.2'                                             as EventTypeId,
        'Arrears chase-  automated' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Customer'                                          as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  m.EventDateTime                                                 as EventDateTime,
                21                                                              as SourceSystemId,
                cast(m.CampaignId as string)                                    as EventSourceReference,
                e.PolicyCode                                                    as SourcePolicyReference,
                m.ClientCode                                                    as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType,
                e.PolicyId
        from    ${ads.ods}.MessageEvent m
        join    (
                select  *, row_number() over (partition by ClientCode, ArrearsStart order by LoanReference) as rn
                from    ${ads.stg}.adsv2_minl_arrears_episode
                ) e     on  e.ClientCode = m.ClientCode and e.rn = 1
                        and cast(m.EventDateTime as date) >= e.ArrearsStart
                        and cast(m.EventDateTime as date) <= e.EpisodeChaseEnd
                        and (m.PolicyCode is null or m.PolicyCode = e.PolicyCode)
        where   m.CampaignName in ('Arrears - Chase 1 - 1 Day Past', 'Arrears - Chase 2 - 7 Days ago', 'Arrears - Chase 3 - 14 Days Past',
                                   'Arrears - Chase 4 - 21 Days Past', 'Arrears - Chase 5 - 28 Days Past')
        and     m.MessageType = 'SMS'
        and     m.InteractionType = 'sent'
        and     m.ClientCode is not null
        and     m.EventDateTime >= timestamp '${ads.start_time}' and m.EventDateTime < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## ARR.O1: Arrears cleared


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_arr_o1  (ARR.O1)                                                        */
/*  Description : Arrears / Outcome. D-E1.9: arrears cleared = the loan returns from arrears (5) to act */
/*              : ive (6), at the first load day active again.                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyLoanStatus                             */
/*              : Reads: stg.adsv2_minl_arrears_episode                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_arr_o1 as
select  xxhash64('ARR.O1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'ARR.O1'                                            as EventTypeId,
        'Arrears cleared' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(NextFromDate as timestamp)                                 as EventDateTime,
                24                                                              as SourceSystemId,
                LoanReference                                                   as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType,
                PolicyId
        from    ${ads.stg}.adsv2_minl_arrears_episode
        where   NextLoanStatusId = 6
        and     cast(NextFromDate as timestamp) >= timestamp '${ads.start_time}' and cast(NextFromDate as timestamp) < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## ARR.3: Loan defaults-  arrears not cleared


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_arr_3  (ARR.3)                                                          */
/*  Description : Arrears / Spine. D-E1.9: loan defaults = arrears (5) moves to default (4), at the fir */
/*              : st load day in default. 04 used the Chase 5 SMS. Loans that go from active straight t */
/*              : o default (6 > 4, about 22k day-to-day changes) start no arrears episode.             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyLoanStatus                             */
/*              : Reads: stg.adsv2_minl_arrears_episode                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_arr_3 as
select  xxhash64('ARR.3', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'ARR.3'                                             as EventTypeId,
        'Loan defaults-  arrears not cleared' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(NextFromDate as timestamp)                                 as EventDateTime,
                24                                                              as SourceSystemId,
                LoanReference                                                   as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType,
                PolicyId
        from    ${ads.stg}.adsv2_minl_arrears_episode
        where   NextLoanStatusId = 4
        and     cast(NextFromDate as timestamp) >= timestamp '${ads.start_time}' and cast(NextFromDate as timestamp) < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## ARR.O2: Escalation to forced cancellation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_arr_o2  (ARR.O2)                                                        */
/*  Description : Arrears / Outcome. D-E1.9: escalation to forced cancellation = the defaulted loan (4, */
/*              :  entered from arrears) moves to cancelled / terminated (3), at the first load day can */
/*              : celled; hands off to C1b. 04 took any Relay 'Insurer Led%' event on a policy that eve */
/*              : r had an arrears campaign.                                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyLoanStatus                             */
/*              : Reads: stg.adsv2_minl_arrears_episode                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_arr_o2 as
select  xxhash64('ARR.O2', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'ARR.O2'                                            as EventTypeId,
        'Escalation to forced cancellation' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(Next2FromDate as timestamp)                                as EventDateTime,
                24                                                              as SourceSystemId,
                LoanReference                                                   as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType,
                PolicyId
        from    ${ads.stg}.adsv2_minl_arrears_episode
        where   NextLoanStatusId = 4
        and     Next2LoanStatusId = 3
        and     cast(Next2FromDate as timestamp) >= timestamp '${ads.start_time}' and cast(Next2FromDate as timestamp) < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## CL1a: Motor claim reported


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_cl1a  (CL1a)                                                            */
/*  Description : Claims / Spine. Notebook 04 cell 309 rebuilt (D-E1.11): inbound calls wrap-up coded ' */
/*              : Claims' on any queue (not QD_*), plus calls whose type is unknown (the merged 'Claims */
/*              :  / Windscreen / Breakdown' code from 2026-04, or uncoded INBOUND_Claims calls), by a  */
/*              : caller with an open Motor policy; one event per conversation. 04 matched INBOUND_Clai */
/*              : ms callers to Motor policies by phone (stg.Motor_PhoneNumbers, not built in 04), one  */
/*              : event per policy for the whole window.                                                */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_cl1a as
select  xxhash64('CL1a', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'CL1a'                                              as EventTypeId,
        'Motor claim reported' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   CallerHasMotor = 1
        and     (CodeClaims = 1 or CodeClaimsMerged = 1
                 or (OnClaims = 1 and ClaimsQueueCoded = 0 and CodeBreakdown = 0 and CodeWindscreen = 0))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## CL1b: Breakdown assistance requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_cl1b  (CL1b)                                                            */
/*  Description : Claims / Spine. New (D-E1.11). Inbound calls wrap-up coded 'Breakdown' on any queue ( */
/*              : not QD_*) by a caller with an open Motor policy.                                      */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_cl1b as
select  xxhash64('CL1b', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'CL1b'                                              as EventTypeId,
        'Breakdown assistance requested' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   CallerHasMotor = 1
        and     CodeBreakdown = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## CL1c: Windscreen claim initiated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_cl1c  (CL1c)                                                            */
/*  Description : Claims / Spine. New (D-E1.11). Inbound calls wrap-up coded 'Windscreen' on any queue  */
/*              : (not QD_*) by a caller with an open Motor policy.                                     */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_cl1c as
select  xxhash64('CL1c', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'CL1c'                                              as EventTypeId,
        'Windscreen claim initiated' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   CallerHasMotor = 1
        and     CodeWindscreen = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## CL3.F1: Nobody tells the customer anything


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_claim_call                                                             */
/*  Description : Motor claims calls (the CL1a / CL1b / CL1c population) with the caller's phone key.   */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_claim_call as
select  ConversationId, CallStart, CallerPhoneKey, MotorPolicyCode, MotorPolicyId
from    ${ads.stg}.adsv2_minl_motor_call
where   CallerHasMotor = 1
and     (CodeClaims = 1 or CodeClaimsMerged = 1 or CodeBreakdown = 1 or CodeWindscreen = 1
         or (OnClaims = 1 and ClaimsQueueCoded = 0))

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_cl3_f1  (CL3.F1)                                                        */
/*  Description : Claims / Friction. New (D-E1.13). A further claims call (CL1a / b / c) by the same ca */
/*              : ller within 30 days of an earlier one: the first is the notification, the rest chasin */
/*              : g. The caller is the phone key (ContactScvLink not built yet).                        */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_claim_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_cl3_f1 as
select  xxhash64('CL3.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'CL3.F1'                                            as EventTypeId,
        'Nobody tells the customer anything' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    (
                select  *, lag(CallStart) over (partition by CallerPhoneKey order by CallStart, ConversationId) as PrevCallStart
                from    ${ads.stg}.adsv2_minl_claim_call
                where   CallerPhoneKey is not null
                ) c
        where   PrevCallStart is not null
        and     PrevCallStart >= CallStart - interval 30 days
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## CL4: Policy continues with a claim on it


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_cl4  (CL4)                                                              */
/*  Description : Claims / Spine. New (D-E1.11). A recorded Motor claim dated inside its own policy's t */
/*              : erm (a claim during Chill cover; earlier claims are declared history and stay on R0.F */
/*              : 2), one event per claim at the claim (incident) date. The term is the sale's year to  */
/*              : its renew date (PolicySale).                                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyClaimMotor, PolicySale                 */
/*              : Reads: ods_v2.PolicyClaimMotor, ods_v2.PolicySale                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_cl4 as
select  xxhash64('CL4', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'CL4'                                               as EventTypeId,
        'Policy continues with a claim on it' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(c.ClaimDate as timestamp)                                  as EventDateTime,
                1                                                               as SourceSystemId,
                concat_ws('|', cast(c.ClaimSlot as string), cast(c.ClaimDateRaw as string), c.ClaimType) as EventSourceReference,
                c.PolicyCode                                                    as SourcePolicyReference,
                max(c.PolicyId)                                                 as PolicyId
        from    ${ads.ods}.PolicyClaimMotor c
        join    ${ads.ods}.PolicySale s         on s.PolicyCode = c.PolicyCode
        where   s.PolicyRenewDateAdj is not null
        and     c.ClaimDate >= add_months(cast(s.PolicyRenewDateAdj as date), -12)
        and     c.ClaimDate <  cast(s.PolicyRenewDateAdj as date)
        and     cast(c.ClaimDate as timestamp) >= timestamp '${ads.start_time}' and cast(c.ClaimDate as timestamp) < timestamp '${ads.end_time}'
        group by c.PolicyCode, c.ClaimSlot, c.ClaimDateRaw, c.ClaimType, c.ClaimDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## CL.O1: Renewed with the claim priced in


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_claim_cycle                                                            */
/*  Description : Renewal cycles (the shared renewal base) whose ending term holds a CL4 claim.         */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.PolicyClaimMotor                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_claim_cycle as
select  distinct b.PolicyCode, b.RenewalDate, b.Renewed, b.Lost, b.OutcomeFinal, b.RenewedDate, b.RenewedPolicyCode, b.CancelDate
from    ${ads.stg}.adsv2_renewal_base b
join    ${ads.ods}.PolicyClaimMotor c   on c.PolicyCode = b.PolicyCode
where   b.LYPolicyTypeGroup = 'Motor'
and     c.ClaimDate >= add_months(b.RenewalDate, -12)
and     c.ClaimDate <  b.RenewalDate

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_cl_o1  (CL.O1)                                                          */
/*  Description : Claims / Outcome. New (D-E1.11). The claimed term's renewal (shared renewal cycles, D */
/*              : -E1.19 / 21) renewed inside the window: one event per cycle at the renewal date of th */
/*              : e new term.                                                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyClaimMotor, PolicySale, PolicyRenewal  */
/*              : Reads: stg.adsv2_minl_claim_cycle                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_cl_o1 as
select  xxhash64('CL.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'CL.O1'                                             as EventTypeId,
        'Renewed with the claim priced in' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(RenewedDate as timestamp)                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_minl_claim_cycle
        where   Renewed = 1
        and     Lost = 0
        and     RenewedDate is not null
        and     cast(RenewedDate as timestamp) >= timestamp '${ads.start_time}' and cast(RenewedDate as timestamp) < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## CL.O2: Policy ended


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_cl_o2  (CL.O2)                                                          */
/*  Description : Claims / Outcome. New (D-E1.11). The claimed term's cycle not renewed inside the wind */
/*              : ow (final outcomes only): at the cancellation inside the term, else the due date.     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicyClaimMotor, PolicySale, PolicyRenewal  */
/*              : Reads: stg.adsv2_minl_claim_cycle                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_cl_o2 as
select  xxhash64('CL.O2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'CL.O2'                                             as EventTypeId,
        'Policy ended' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(coalesce(CancelDate, RenewalDate) as timestamp)            as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_minl_claim_cycle
        where   Lost = 1
        and     OutcomeFinal = 1
        and     cast(coalesce(CancelDate, RenewalDate) as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(CancelDate, RenewalDate) as timestamp) < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## D1: Customer rings and asks for a document


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_d1  (D1)                                                                */
/*  Description : Doc Requests / Spine. Notebook 04 cell 310 rebuilt (D-E1.5): each inbound Documents_O */
/*              : ut call by a caller with an open Motor policy, coded with a document request or uncod */
/*              : ed (transfers and other topics are not D1; Home-named codes go to Home), at the call  */
/*              : start. 04 kept only calls followed by a Relay document within 2 days, Jul-Aug 2026 on */
/*              : ly.                                                                                   */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_d1 as
select  xxhash64('D1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'D1'                                                as EventTypeId,
        'Customer rings and asks for a document' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   OnDocsOut = 1
        and     CallerHasMotor = 1
        and     DocsHomeCode = 0
        and     (DocsMotorCode = 1 or DocsCoded = 0)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## D1.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_d1_f1  (D1.F1)                                                          */
/*  Description : Doc Requests / Friction. Notebook 04 cell 311 rebuilt (D-E1.5, D-E1.25). D1 calls tha */
/*              : t waited more than 600 seconds on Documents_Out or were abandoned there.              */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_d1_f1 as
select  xxhash64('D1.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'D1.F1'                                             as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_motor_call
        where   OnDocsOut = 1
        and     CallerHasMotor = 1
        and     DocsHomeCode = 0
        and     (DocsMotorCode = 1 or DocsCoded = 0)
        and     (DocsWaitMs > 600000 or DocsAbandoned > 0)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## D3.F1: Non-arrival-  and the customer rings again


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_minl_doc_call                                                               */
/*  Description : Motor document-request calls (the D1 population) with the caller's phone key and requ */
/*              : est type.                                                                             */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_minl_motor_call                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_minl_doc_call as
select  ConversationId, CallStart, CallerPhoneKey, DocRequestType, MotorPolicyCode, MotorPolicyId
from    ${ads.stg}.adsv2_minl_motor_call
where   OnDocsOut = 1
and     CallerHasMotor = 1
and     DocsHomeCode = 0
and     (DocsMotorCode = 1 or DocsCoded = 0)

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_d3_f1  (D3.F1)                                                          */
/*  Description : Doc Requests / Friction. Notebook 04 cell 313 rebuilt (D-E1.8): a D1 call by the same */
/*              :  caller within 14 days of an earlier D1 (the same request type where both are coded), */
/*              :  at the repeat call. 04 counted clients with calls on more than one date, any time ap */
/*              : art.                                                                                  */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_doc_call                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_d3_f1 as
select  xxhash64('D3.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'D3.F1'                                             as EventTypeId,
        'Non-arrival-  and the customer rings again' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  c.CallStart                                                     as EventDateTime,
                20                                                              as SourceSystemId,
                c.ConversationId                                                as EventSourceReference,
                max(c.MotorPolicyCode)                                          as SourcePolicyReference,
                c.CallerPhoneKey                                                as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                max(c.MotorPolicyId)                                            as PolicyId
        from    ${ads.stg}.adsv2_minl_doc_call c
        join    ${ads.stg}.adsv2_minl_doc_call p on p.CallerPhoneKey = c.CallerPhoneKey
                                                and p.ConversationId <> c.ConversationId
                                                and p.CallStart < c.CallStart
                                                and p.CallStart >= c.CallStart - interval 14 days
                                                and (p.DocRequestType is null or c.DocRequestType is null
                                                     or p.DocRequestType = c.DocRequestType)
        where   c.CallerPhoneKey is not null
        group by c.ConversationId, c.CallStart, c.CallerPhoneKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## D.V2: Proof of no claims bonus


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_d_v2  (D.V2)                                                            */
/*  Description : Doc Requests / Spine. New (D-E1.6). D1 calls wrap-up coded as a no claims bonus reque */
/*              : st ('NCB Request', 'Docs Out - NCB Request'; codes from 2024-01).                     */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_doc_call                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_d_v2 as
select  xxhash64('D.V2', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'D.V2'                                              as EventTypeId,
        'Proof of no claims bonus' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_doc_call
        where   DocRequestType = 'D.V2'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## D.V3: Letter of indemnity


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_d_v3  (D.V3)                                                            */
/*  Description : Doc Requests / Spine. New (D-E1.6). D1 calls coded as a letter of indemnity request ( */
/*              : the Motor and unlabelled LOI codes; LOI Home codes go to Home).                       */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_doc_call                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_d_v3 as
select  xxhash64('D.V3', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'D.V3'                                              as EventTypeId,
        'Letter of indemnity' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_doc_call
        where   DocRequestType = 'D.V3'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## D.V4: Direct Debit schedule or payment evidence


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_d_v4  (D.V4)                                                            */
/*  Description : Doc Requests / Spine. New (D-E1.6). D1 calls coded 'CPF (Active) Payment Date/Amount  */
/*              : Query' (read as DD schedule / payment evidence: an assumption, listed for the client) */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_doc_call                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_d_v4 as
select  xxhash64('D.V4', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'D.V4'                                              as EventTypeId,
        'Direct Debit schedule or payment evidence' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_doc_call
        where   DocRequestType = 'D.V4'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## D.V5: Named driver evidence


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_d_v5  (D.V5)                                                            */
/*  Description : Doc Requests / Spine. New (D-E1.6). D1 calls coded 'LONDE Request' (read as a letter  */
/*              : of named driver experience: an assumption, listed for the client).                    */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_doc_call                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_d_v5 as
select  xxhash64('D.V5', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'D.V5'                                              as EventTypeId,
        'Named driver evidence' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_doc_call
        where   DocRequestType = 'D.V5'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## D.V6: Copy policy documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_d_v6  (D.V6)                                                            */
/*  Description : Doc Requests / Spine. New (D-E1.6). D1 calls coded as a copy of the policy documents  */
/*              : (CS Documents Out, Cover Confirmation, Duplicate Pack, MTA Docs (Motor)).             */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, SourceCustomer, PolicyMonthEnd    */
/*              : Reads: stg.adsv2_minl_doc_call                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_d_v6 as
select  xxhash64('D.V6', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'D.V6'                                              as EventTypeId,
        'Copy policy documents' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CallStart                                                       as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                MotorPolicyCode                                                 as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType,
                MotorPolicyId                                                   as PolicyId
        from    ${ads.stg}.adsv2_minl_doc_call
        where   DocRequestType = 'D.V6'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

In [ ]:
%%pyspark
# Events per family (a family with no events is reported, not failed: check its source and window)
for t in ["adsv2_evt_m1", "adsv2_evt_m1_f1", "adsv2_evt_m_v1", "adsv2_evt_m_v2", "adsv2_evt_m_v3", "adsv2_evt_m_v4", "adsv2_evt_m3", "adsv2_evt_m4", "adsv2_evt_m4a", "adsv2_evt_m6", "adsv2_evt_m3_b1_1", "adsv2_evt_m3_b1_2", "adsv2_evt_m3_b1_4", "adsv2_evt_m3_b1_5", "adsv2_evt_m3_b1_3", "adsv2_evt_m3_b1_6", "adsv2_evt_c1a", "adsv2_evt_c1b", "adsv2_evt_c1a_f1", "adsv2_evt_c2", "adsv2_evt_c3", "adsv2_evt_c4", "adsv2_evt_c5", "adsv2_evt_c6", "adsv2_evt_arr_1", "adsv2_evt_arr_2", "adsv2_evt_arr_o1", "adsv2_evt_arr_3", "adsv2_evt_arr_o2", "adsv2_evt_cl1a", "adsv2_evt_cl1b", "adsv2_evt_cl1c", "adsv2_evt_cl3_f1", "adsv2_evt_cl4", "adsv2_evt_cl_o1", "adsv2_evt_cl_o2", "adsv2_evt_d1", "adsv2_evt_d1_f1", "adsv2_evt_d3_f1", "adsv2_evt_d_v2", "adsv2_evt_d_v3", "adsv2_evt_d_v4", "adsv2_evt_d_v5", "adsv2_evt_d_v6"]:
    n = spark.table(f"{stg_schema}.{t}").count()
    print(f"{t:<40} {n:>12,}" + ("   <- no events" if n == 0 else ""))
ads_run_log("15_Motor_InLife", "succeeded")
